# Mission EdStats — qualité des données

**Contexte.** Mark, notre manager, nous a confié une première mission d'analyse exploratoire : déterminer si les données de la Banque mondiale sur l'éducation (EdStats) peuvent éclairer le projet d'**expansion internationale d'academy**. Il veut explorer les pays à fort potentiel de clients pour nos services, et voir comment ce potentiel pourrait évoluer.

**Ce notebook.** Avant de chercher des pays, on vérifie la qualité des 5 fichiers de données, l'un après l'autre : ce que représente une ligne, combien de lignes et de colonnes, s'il y a des doublons, quelles valeurs sont manquantes, quelles colonnes garder, et les premières statistiques. Chaque fichier aura sa propre section, traitée l'une après l'autre.

## Mise en place

### 1. Imports et dossier des données

`RESOURCES_DIR` pointe vers le dossier `resources/` contenant les 5 CSV. Selon l'endroit d'où le notebook est lancé, le chemin relatif change : on teste plusieurs candidats et on garde le premier qui existe.

In [1]:
import pandas as pd
from pathlib import Path

candidates = [
    Path.cwd() / "notebooks" / "resources",  # lancé depuis la racine du projet
    Path.cwd() / "resources",                 # lancé depuis le dossier notebooks/
]

RESOURCES_DIR = None
for candidate in candidates:
    if candidate.exists():
        RESOURCES_DIR = candidate.resolve()  # chemin absolu
        break

if RESOURCES_DIR is None:
    raise FileNotFoundError("Dossier resources/ introuvable")

RESOURCES_DIR

PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/p2-education-systems-analysis-ph1-2-01/notebooks/resources')

### 2. La fonction de chargement brut

`pd.read_csv(...)` lit le CSV et renvoie le DataFrame. Deux réglages, spécifiques à ces fichiers :

- `encoding="utf-8"` : précise l'encodage du texte (pour bien lire les accents).
- `usecols=lambda column: "Unnamed" not in column` : écarte les colonnes « Unnamed ». Chaque ligne du CSV se termine par une virgule, ce qui crée une colonne vide `Unnamed: 0` — un artefact du fichier, pas une vraie donnée.

**Pourquoi on charge « brut », sans nettoyage** (bonne pratique aussi en entreprise) :

- pas de `fillna(...)` : remplacer les valeurs manquantes (les `NaN`) les rendrait invisibles. Or on veut justement les *compter* ensuite avec `isnull()`.
- pas de `astype(object)` : forcer les colonnes en « texte générique » détruirait les types, et les statistiques (`describe()`) ne fonctionneraient plus.

Les données restent telles quelles : les calculs seront reproductibles, et l'aperçu des données (notebook `data_overview.ipynb`) reste fidèle aux fichiers d'origine.

In [2]:
def load_csv_analysis(name):
    """Lit le CSV `name` depuis RESOURCES_DIR, tel quel (brut), pour l'analyse."""
    return pd.read_csv(
        RESOURCES_DIR / name,
        encoding="utf-8",
        usecols=lambda column: "Unnamed" not in column,
    )

### 3. Chargement des 5 CSV

Les 5 fichiers sont chargés dans un dictionnaire `dfs` (`dfs["EdStatsCountry.csv"]` = le DataFrame de ce fichier). Ordre : les référentiels d'abord (pays, indicateurs), puis la table de mesures, puis les tables de liaison.

`df.shape` renvoie `(lignes, colonnes)`.

*Note : `EdStatsData.csv` fait environ 326 Mo, cette cellule peut prendre une minute.*

In [3]:
FICHIERS = [
    "EdStatsCountry.csv",        # référentiel des pays et agrégats
    "EdStatsSeries.csv",         # dictionnaire des indicateurs
    "EdStatsData.csv",           # mesures par pays, indicateur et année
    "EdStatsCountry-Series.csv", # sources par couple pays × indicateur
    "EdStatsFootNote.csv",       # annotations par pays, indicateur et année
]

dfs = {}
for name in FICHIERS:
    dfs[name] = load_csv_analysis(name)
    print(f"{name} : {dfs[name].shape[0]} lignes, {dfs[name].shape[1]} colonnes")

EdStatsCountry.csv : 241 lignes, 31 colonnes
EdStatsSeries.csv : 3665 lignes, 20 colonnes


EdStatsData.csv : 886930 lignes, 69 colonnes
EdStatsCountry-Series.csv : 613 lignes, 3 colonnes
EdStatsFootNote.csv : 643638 lignes, 4 colonnes


### 4. Récapitulatif des dimensions (lignes × colonnes), à partir des `shape` :

In [4]:
recap = pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in dfs.items()],
    columns=["Fichier", "Lignes", "Colonnes"],
)
recap

,Fichier,Lignes,Colonnes
0,EdStatsCountry.csv,241,31
1,EdStatsSeries.csv,3665,20
2,EdStatsData.csv,886930,69
3,EdStatsCountry-Series.csv,613,3
4,EdStatsFootNote.csv,643638,4


Aperçu brut du référentiel des pays (pas de colonne `Unnamed`, les valeurs manquantes encore en `NaN`) :

In [5]:
dfs["EdStatsCountry.csv"].head(3)

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,SNA data for 2000-2011 are updated from offici...,Latin America & Caribbean,High income: nonOECD,AW,...,NaN,NaN,2010,NaN,NaN,Yes,NaN,NaN,2012.0,NaN
1,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,AF,...,Consolidated central government,General Data Dissemination System (GDDS),1979,"Multiple Indicator Cluster Survey (MICS), 2010/11","Integrated household survey (IHS), 2008",NaN,2013/14,NaN,2012.0,2000
2,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Upper middle income,AO,...,Budgetary central government,General Data Dissemination System (GDDS),1970,"Malaria Indicator Survey (MIS), 2011","Integrated household survey (IHS), 2008",NaN,2015,NaN,NaN,2005


### Bilan de la mise en place

Les 5 CSV sont chargés bruts dans `dfs`, prêts pour le protocole qualité. Prochaine étape : analyser chaque fichier l'un après l'autre, en commençant par les référentiels (`EdStatsCountry`, `EdStatsSeries`), puis la table de mesures (`EdStatsData`), puis les tables de liaison (`EdStatsCountry-Series`, `EdStatsFootNote`), avant la synthèse destinée à Mark.